In [7]:
%load_ext autoreload
%autoreload 2
%reset -f

[autoreload of lib.KPIHubConnection failed: Traceback (most recent call last):
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 475, in superreload
    module = reload(module)
  File "/usr/local/lib/python3.9/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 613, in _exec
  File "<frozen importlib._bootstrap_external>", line 850, in exec_module
  File "<frozen importlib._bootstrap>", line 228, in _call_with_frames_removed
  File "/home/sandbox/personal-repos/KPIHub/lib/KPIHubConnection.py", line 3, in <module>
    from config import *
ModuleNotFoundError: No module named 'config'
]


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
from pathlib import Path
import os
from os.path import join
import sys

# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

In [9]:
LOG_PATH = os.path.join(INGESTER_LOG_PATH, 'ReportSummaryIngester.log')
Logger = Loggers(logger_name = 'ReportSummaryIngester', keys = ['File', 'Slack'])
Logger.clear_handlers()
import logging

file_handler = logging.FileHandler(LOG_PATH)
# Set date format to dd-mm-yyyy in log output
formatter = logging.Formatter('%(asctime)s [%(levelname)s] %(name)s: %(message)s', datefmt='%d-%m-%Y')
file_handler.setFormatter(formatter)
Logger.File.addHandler(file_handler)

slack_handler = logging.StreamHandler(SlackWriter(channel = 'C0B9PGDNHH7'))
Logger.Slack.addHandler(slack_handler)


In [10]:
Logger.info("="*100)
Logger.Slack.info('Starting Report Summary Ingester')

In [11]:
#Get customer list
customer_list = Query(query = "SELECT * FROM KPI_Customer WHERE DBLocation IS NOT 'Unknown'").execute(KPIHub_Conn)

#Set the time window for the update
current_date = date.today()
update_window = current_date - timedelta(days=UPDATE_WINDOW_DAYS)

In [12]:
for _, row in customer_list.iterrows():
    customer_name = row['Name']
    customer_id = row['CustomerId']
    customer_db = row['DBLocation']

    Logger.info(f"Processing customer: {customer_name}")
    # Query to get the number of different ReportId from reports
    query =   f"""SELECT COUNT(DISTINCT R.Id) as ReportCount FROM
        Report R
    LEFT JOIN Customer C ON
        R.CustomerId = C.Id
    LEFT JOIN ReportLabel RL ON
        R.Id = RL.ReportId
    LEFT JOIN Label L ON
        RL.LabelId = L.Id
    LEFT JOIN ReportType ON
        R.ReportTypeId = ReportType.Id
    LEFT JOIN ReportArea RA ON R.Id = RA.ReportId
    LEFT JOIN ReportCompliance RC ON R.Id = RC.ReportId
    LEFT JOIN ReportAreaCovered RAC ON R.Id = RAC.ReportId
    WHERE
        R.CustomerId = '{customer_id}' AND R.DateStarted >= '{STARTING_DATE}' AND L.Title = 'Final Checkbox' AND RL.IsActive = 1
        AND L.Title = 'Final Checkbox'
        AND RL.IsActive = 1
    """

    numReports = Query(query =query).execute(CONN_DICT[customer_db])
    
    num_unique_report_ids = numReports.iloc[0]['ReportCount'] if len(numReports) > 0 else 0
    Logger.info(f"Number of unique Reports in LSDB: {num_unique_report_ids}")

    #Query the last report from the KPIHUb
    query = Query(f"SELECT * FROM KPI_ReportSummary WHERE CustomerId = '{customer_id}' ORDER BY LastUpdated DESC LIMIT 1").execute(KPIHub_Conn)
    if len(query) > 0:
        last_updated = query.iloc[0]['LastUpdated']
        starting_date = update_window
        process_reports = True
        Logger.info(f"Last updated: {last_updated}, processing")
    else:
        Logger.info(f"No reports found, starting from {starting_date}")

        process_reports = True

    if process_reports:
        query = get_reports(customer_name, starting_date=starting_date, final_checkbox = True)
        LSDB_COLS = [
            'ReportId',
            'CustomerId',
            'ReportName',
            'ReportDate',
            'ReportYear',
            'ReportMonth',
            'ReportWeek',
            'ReportAssetLengthKm',
            'AssetCoveredLengthKm',
            'DistributionPipeKm',
            'DistributionPipeCoveredKm',
            'ServicePipeKm',
            'ServicePipeCoveredKm',
        ]

                # Clisify (classify) the report_summary by different periods using to_period: quarter, year, month, week
        #report_summary['ReportQuarter'] = pd.to_datetime(report_summary['ReportDate']).dt.isocalendar().quarter

        DATAHUB_COLS = ['ReportId', 'BoundaryName', 'BoundaryType', 'BoundaryMode', 'BoundaryPlant', 'BoundarySubplant', 'BoundaryRegion', 'BoundarySubRegion']

        reports_lsdb = query.execute(CONN_DICT[customer_db])
        if customer_db == 'EU1' or customer_db == 'EU2':
            reports_lsdb.db.set_query(query_reports_view(report_table = 'temp_reports'))
            reports_datahub = reports_lsdb.db.execute(DATAHUB_Conn, source_col = 'ReportId', temp_table_name = 'temp_reports')
            # Clisify (classify) the report_summary by different periods using to_period: quarter, year, month, week
            #report_summary['ReportQuarter'] = pd.to_datetime(report_summary['ReportDate']).dt.isocalendar().quarter
            reports_lsdb['ReportYear'] = pd.to_datetime(reports_lsdb['ReportDate']).dt.year
            reports_lsdb['ReportMonth'] = pd.to_datetime(reports_lsdb['ReportDate']).dt.month
            reports_lsdb['ReportWeek'] = pd.to_datetime(reports_lsdb['ReportDate']).dt.isocalendar().week
            reports = pd.merge(reports_lsdb[LSDB_COLS], reports_datahub[DATAHUB_COLS], on = 'ReportId', how = 'left')

        else:
            reports['ReportYear'] = pd.to_datetime(reports['ReportDate']).dt.year
            reports['ReportMonth'] = pd.to_datetime(reports['ReportDate']).dt.month
            reports['ReportWeek'] = pd.to_datetime(reports['ReportDate']).dt.isocalendar().week
            reports = reports_lsdb[LSDB_COLS]
        # Add/update the LastUpdated column to the reports DataFrame as current timestamp
        reports['LastUpdated'] = datetime.now()
        Logger.info(f"Reports from LSDB starting from {starting_date}: {len(reports)}")
        KPI_ReportSummary.update_table(arguments = {'db_path': DB_PATH, 'DataFrame': reports, 'PrimaryKey': 'ReportId'})
        df_kpi = Query(query = f"SELECT * FROM KPI_ReportSummary WHERE CustomerId = '{customer_id}'").execute(KPIHub_Conn)
        Logger.info(f"Total reports from KPI_ReportSummary: {len(df_kpi)}")
